# Title


**IMDB Movie Review Sentiment Analysis**

# Problem Statement



Nowadays, people share thousands of movie reviews online. These reviews contain different opinions, such as positive and negative feedback.

Manually reading and identifying whether a review is positive or negative takes a lot of time.

Therefore, the problem is to develop a Machine Learning model that automatically analyzes movie reviews and classifies them as Positive or Negative based on the text.

# Main Objective

1. Collect movie reviews from the IMDB dataset.

2. Clean and preprocess the review text.

3. Convert text into numerical features using TF-IDF.

4. Train different Machine Learning classification algorithms.

5. Compare their performance using accuracy.

6. Select the best-performing model using hyperparameter tuning.

7. Predict whether a new movie review is Positive or Negative.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd
import numpy as np
import re
import html
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,BaggingClassifier,AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [72]:
df = pd.read_csv('/content/drive/MyDrive/ML/imdb sentiment analysis/IMDB Dataset.csv')

In [73]:
# Display first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [74]:

# Check number of rows and columns
print("Shape:", df.shape)

Shape: (50000, 2)


In [75]:
# Check column names
print("Columns:", df.columns.tolist())

Columns: ['review', 'sentiment']


In [76]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


**Data cleaning**

In [77]:
df.isnull().sum()

,0
review,0
sentiment,0


**Observation**

There is no missing content


**Check duplicate rows**

In [78]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [79]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

**Observation**

There are totla 418 duplicate values are available so then i remove that values

In [80]:
print(df['sentiment'].value_counts())
print(df['sentiment'].unique())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64
['positive' 'negative']


In [81]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
stop_words -= {'no', 'nor', 'not'}

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # 1. Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # 2. Decode HTML entities
    text = html.unescape(text)

    # 3. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Convert to lowercase
    text = text.lower()

    # 6. Expand common contractions
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "cannot", text)
    text = re.sub(r"\bshan't\b", "shall not", text)
    text = re.sub(r"\bain't\b", "is not", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"'re\b", " are", text)
    text = re.sub(r"'ve\b", " have", text)
    text = re.sub(r"'ll\b", " will", text)
    text = re.sub(r"'m\b", " am", text)
    text = re.sub(r"'d\b", " would", text)

    # 7. Remove punctuation, digits and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Remove extra spaces and tokenize
    tokens = text.split()

    # 9. Remove stopwords while preserving negations
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 10. Lemmatization
    tokens = [
                lemmatizer.lemmatize(word, pos='v')
                for word in tokens
             ]

    # 11. Return processed text
    return ' '.join(tokens)

In [82]:
df['processed_review'] = df['review'].apply(preprocess_text)

In [83]:
df[['review', 'processed_review', 'sentiment']].head()

,review,processed_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewers mention watch oz episode hook ri...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production film technique una...,positive
2,I thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...,positive
3,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stun fi...,positive


**Separate features and target**

In [84]:
X = df['processed_review']
y = df['sentiment']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (49582,)
Target shape: (49582,)


**Encode sentiment labels**

In [85]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['negative' 'positive']


**Observation:**

Convert positive and negative labels into numerical values.

**Split the dataset**

In [86]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


**Observation:**

We'll use 80% for training and 20% for testing, preserving the sentiment proportions with stratify=y.

**TF-IDF Vectorization**

In [87]:
tfidf = TfidfVectorizer()
X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)

# Model Building

**Base Model**

In [22]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

**Prediction**

In [23]:
y_pred = lr.predict(X_test)

**Validation**

In [24]:
logistic_accuracy = accuracy_score(y_test, y_pred)
print('Logistic Regression Accuracy Score : ', logistic_accuracy)

Logistic Regression Accuracy Score :  0.8928103257033377


**Support Vector Classifier**

In [91]:
svc= LinearSVC(max_iter=3000)
svc.fit(X_train, y_train)


LinearSVC(max_iter=3000)

**Prediction**

In [92]:
y_pred = svc.predict(X_test)

In [93]:
# Evaluate
svc_accuracy = accuracy_score(y_test, y_pred)
print('SVC Accuracy Score : ', svc_accuracy)

SVC Accuracy Score :  0.8943228799031965


**Multinomial Naive Bayes Classifier**

In [28]:
nb = MultinomialNB()
nb.fit(X_train, y_train)

MultinomialNB()

In [29]:
# predict
y_pred = nb.predict(X_test)


In [30]:
# Evaluate
MultinomialNB_accuracy = accuracy_score(y_test, y_pred)
print('MultinomialNB Accuracy Score :',MultinomialNB_accuracy)

MultinomialNB Accuracy Score : 0.862861752546133


**Decision Tree**

In [34]:
db = DecisionTreeClassifier()
db.fit(X_train, y_train)

DecisionTreeClassifier()

In [35]:
# predict
y_pred = db.predict(X_test)


In [36]:
# Evaluate
decision_tree_accuracy = accuracy_score(y_test, y_pred)
print('Decision Tree Accuracy Score :', decision_tree_accuracy)

Decision Tree Accuracy Score : 0.7131188867601089


**RandomForestClassifier**

In [37]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [38]:
# predict
y_pred = rf.predict(X_test)


In [39]:
# Evaluate
random_forest_accuracy = accuracy_score(y_test, y_pred)
print('Random Forest Classifier Accuracy Score :', random_forest_accuracy)

Random Forest Classifier Accuracy Score : 0.8515680145205203


**BaggingClassifier**

In [40]:
bc = BaggingClassifier()
bc.fit(X_train, y_train)

BaggingClassifier()

In [41]:
# predict
y_pred = bc.predict(X_test)

In [43]:
# Evaluate
bagging_classifier_accuracy = accuracy_score(y_test, y_pred)
print('Bagging Calssifier Accuracy Score : ', bagging_classifier_accuracy)

Bagging Calssifier Accuracy Score :  0.7837047494201875


**AdaBoostClassifier**

In [44]:
ad = AdaBoostClassifier()
ad.fit(X_train, y_train)

AdaBoostClassifier()

In [45]:
# predict
y_pred = ad.predict(X_test)

In [46]:
# Evaluate
adboost_accuracy = accuracy_score(y_test, y_pred)
print('Adaboost Classifer Accuracy Score : ', adboost_accuracy)

Adaboost Classifer Accuracy Score :  0.7434708077039427


**Gradient Boost Classifier**

In [47]:
gd = GradientBoostingClassifier()
gd.fit(X_train, y_train)

GradientBoostingClassifier()

In [48]:
# Predict
y_pred = gd.predict(X_test)

In [49]:
# Evaluate
gradient_boost_accuracy = accuracy_score(y_test, y_pred)
print('Gradient Boost Classifier Accuracy Score :', gradient_boost_accuracy)

Gradient Boost Classifier Accuracy Score : 0.8117374205909045


In [50]:
df = pd.DataFrame({
                  'Algorithm': [
                              'Logistic Regression',
                              'SVC',
                              'MultinomialNB',
                              'Decision Tree',
                              'Random Forest',
                              'Bagging Classifier',
                              'AdaBoost Classifier',
                              'Gradient Boost Classifier'
                              ],

                  'Accuracy_Score': [
                              logistic_accuracy,
                              svc_accuracy,
                              MultinomialNB_accuracy,
                              decision_tree_accuracy,
                              random_forest_accuracy,
                              bagging_classifier_accuracy,
                              adboost_accuracy,
                              gradient_boost_accuracy
                              ]
                  })



In [54]:
df = df.sort_values(by='Accuracy_Score', ascending= False).reset_index(drop=True)
df['Accuracy_Score'] = df['Accuracy_Score']*100
df

,Algorithm,Accuracy_Score
0,SVC,89.432288
1,Logistic Regression,89.281033
2,MultinomialNB,86.286175
3,Random Forest,85.156801
4,Gradient Boost Classifier,81.173742
5,Bagging Classifier,78.370475
6,AdaBoost Classifier,74.347081
7,Decision Tree,71.311889


**Observation:**

By Experimenting all algorithms Support Vector Classifier got highest accuracy Score then i choose svc has the final model.

# Hyperparameter Tuning for SVC

In [94]:
# Defining Svc model
svc = LinearSVC()

In [95]:
# defining parameters
param_grid = {
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'loss': ['hinge', 'squared_hinge'],
            'class_weight': [None, 'balanced'],
            'tol': [1e-3, 1e-4]
            }

# GridSearchCv
gs = GridSearchCV(
                  estimator=svc,
                  param_grid=param_grid,
                  cv=3,
                  scoring='accuracy',
                  n_jobs=-1,
                  verbose=1
                )

gs.fit(X_train, y_train)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


GridSearchCV(cv=3, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'loss': ['hinge', 'squared_hinge'],
                         'tol': [0.001, 0.0001]},
             scoring='accuracy', verbose=1)

In [96]:
print('Best Parameters:', gs.best_params_)
print('Best Accuracy:', gs.best_score_)

Best Parameters: {'C': 1, 'class_weight': 'balanced', 'loss': 'hinge', 'tol': 0.001}
Best Accuracy: 0.8935584446900395


In [98]:
best_model = gs.best_estimator_

y_pred = best_model.predict(X_test)

In [101]:
final_model_accuracy = accuracy_score(y_test, y_pred)
print('Final Model Accuracy Score :',final_model_accuracy )

Final Model Accuracy Score : 0.8982555208228294


In [102]:
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))


Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.89      0.90      4940
    positive       0.89      0.91      0.90      4977

    accuracy                           0.90      9917
   macro avg       0.90      0.90      0.90      9917
weighted avg       0.90      0.90      0.90      9917



**Save your best model**

In [105]:
import joblib

joblib.dump(best_model, 'final_model.pkl')

joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']

**Predict sentiment for new reviews**

In [109]:
review = ["This movie was fantastic and the acting was excellent"]

v_review = tfidf.transform(review)
prediction = best_model.predict(v_review)

print("Predicted sentiment:", le.inverse_transform(prediction)[0])

Predicted sentiment: positive
